# <span style="color:red"><center>Construcción automática de texto (novela en español)</center></span>

## Introducción

En este cuaderno se entrena una RNN (LSTM bidireccional) para **generar texto automáticamente** a partir de una novela en español almacenada como PDF (`../Datos/novela.pdf`).

Flujo:
1. Extraer el texto del PDF con `pypdf`.
2. Guardar el texto extraído como `../Datos/novela.txt`.
3. Preprocesar (minúsculas, dividir en oraciones).
4. Construir n-gramas y entrenar la red.
5. Generar texto a partir de una semilla.

## Referencias

1. Laurencemoroney blog: https://storage.googleapis.com/laurencemoroney-blog.appspot.com/sonnets.txt
2. pypdf: https://pypdf.readthedocs.io/

## Contenido
* [Librerías](#Librerías)
* [Lee el PDF y guárdalo como TXT](#Lee-el-PDF)
* [Preprocesamiento](#Preprocesamiento)
* [Tokeniza el texto](#Tokeniza)
* [Crea n-Grams](#n-grams)
* [Crea el modelo](#Modelo)
* [Entrena](#Entrena)
* [Genera texto automáticamente](#Genera)

## <span style="color:blue">Librerías</span>

In [25]:
!pip install -q pypdf

In [26]:
import os
import re
import numpy as np

from pypdf import PdfReader

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras import regularizers
from tensorflow.keras.utils import to_categorical, plot_model

import tensorflow as tf
print('TensorFlow:', tf.__version__)

TensorFlow: 2.17.1


## <span style="color:blue">Lee el PDF y guárdalo como TXT</span>

Se lee `../Datos/MVLL.pdf`, se extrae el texto página a página y se concatena. Luego se guarda como `../Datos/novela.txt`.

In [27]:
PDF_PATH = '../Datos/mvll_lituma.pdf'
TXT_PATH = '../Datos/mvll_lituma.txt'

reader = PdfReader(PDF_PATH)
num_pages = len(reader.pages)
print(f'Páginas encontradas: {num_pages}')

texto_completo = []
for i, page in enumerate(reader.pages):
    texto_pagina = page.extract_text() or ''
    texto_completo.append(texto_pagina)
    if (i + 1) % 20 == 0:
        print(f'  procesadas {i+1}/{num_pages} páginas...')

texto_completo = '\n'.join(texto_completo)
print(f'Caracteres extraídos: {len(texto_completo):,}')

Páginas encontradas: 254
  procesadas 20/254 páginas...
  procesadas 40/254 páginas...
  procesadas 60/254 páginas...
  procesadas 80/254 páginas...
  procesadas 100/254 páginas...
  procesadas 120/254 páginas...
  procesadas 140/254 páginas...
  procesadas 160/254 páginas...
  procesadas 180/254 páginas...
  procesadas 200/254 páginas...
  procesadas 220/254 páginas...
  procesadas 240/254 páginas...
Caracteres extraídos: 469,546


In [28]:
with open(TXT_PATH, 'w', encoding='utf-8') as f:
    f.write(texto_completo)

print(f'Archivo guardado en: {TXT_PATH}')
print(f'Tamaño: {os.path.getsize(TXT_PATH)/1024:.1f} KB')
print('\nPrimeros 500 caracteres:\n')
print(texto_completo[:500])

Archivo guardado en: ../Datos/mvll_lituma.txt
Tamaño: 476.3 KB

Primeros 500 caracteres:

OceanofPDF.com
Lituma en los Andes
Mario Vargas Llosa
OceanofPDF.com
Título: Lituma en los Andes
© 1994-06, Mario Vargas Llosa
Ilustración de cubierta: El Minotauro (grabado, 1993), de
Pablo Picasso. Diseño Portada: Mario Blanco
Editorial: Editorial Planeta, S.A.
ISBN: 9788408010470
Revisado por: teref
Agradecimientos: eBook Master
Versión: 1.0
 
Reseña:
Retomando a uno de los personajes más
legendarios de sus novelas anteriores, Mario Vargas
Llosa vuelve a la ficción luego de cinco años, con un


## <span style="color:blue">Preprocesamiento</span>

Para una novela no conviene dividir por `\n` (los párrafos serían enormes). Se divide por **signos de puntuación finales** (`. ! ? …`) para obtener oraciones manejables.

Además se limita a `MAX_SENTENCES` para que el entrenamiento sea viable (una novela completa puede generar >100k secuencias y tardar horas). Sube este valor si tu equipo lo soporta.

In [29]:
MAX_SENTENCES = 3000   # ajusta según tu tiempo/equipo (usa None para el texto completo)

data = open(TXT_PATH, encoding='utf-8').read()
data = data.lower()

# Normalizaciones ligeras para español
data = data.replace('\r', ' ').replace('\n', ' ')
data = re.sub(r'\s+', ' ', data).strip()

# Divide en oraciones usando . ! ? … (uno o más signos seguidos de espacio)
sentences = re.split(r'(?<=[.!?…])\s+', data)
sentences = [s.strip() for s in sentences if len(s.strip()) > 0]

if MAX_SENTENCES is not None:
    sentences = sentences[:MAX_SENTENCES]

corpus = sentences
print(f'Número de oraciones: {len(corpus)}')
print('\nPrimeras 5 oraciones:')
for s in corpus[:5]:
    print('  -', s[:120])

Número de oraciones: 3000

Primeras 5 oraciones:
  - oceanofpdf.com lituma en los andes mario vargas llosa oceanofpdf.com título: lituma en los andes © 1994-06, mario vargas
  - diseño portada: mario blanco editorial: editorial planeta, s.a.
  - isbn: 9788408010470 revisado por: teref agradecimientos: ebook master versión: 1.0 reseña: retomando a uno de los person
  - el cabo lituma y su adjunto tomás, destinados en un campamento militar en plena zona minera de las montañas del perú, se
  - la obsesión de ese enigma sin solución se suma a la atmósfera hostil de un mundo primitivo, en donde se diluyen las fron


## <span style="color:blue">Tokeniza el texto</span>

In [30]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index) + 1   # +1 por el índice 0 (padding/OOV)

print('Tamaño del vocabulario (con padding):', total_words)
print('Palabras más frecuentes:')
for w, i in list(tokenizer.word_index.items())[:10]:
    print(f'  {i:>5}  {w}')

Tamaño del vocabulario (con padding): 7657
Palabras más frecuentes:
      1  de
      2  la
      3  y
      4  a
      5  que
      6  el
      7  los
      8  en
      9  se
     10  no


## <span style="color:blue">Crea n-Grams para datos de entrenamiento y etiquetas</span>

In [31]:
input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print('Primeras 5 secuencias:', input_sequences[:5])

max_sequence_len = max(len(x) for x in input_sequences)
print('Longitud del n-grama más largo:', max_sequence_len)
print('Total de secuencias generadas:', len(input_sequences))

Primeras 5 secuencias: [[1160, 1161], [1160, 1161, 28], [1160, 1161, 28, 8], [1160, 1161, 28, 8, 7], [1160, 1161, 28, 8, 7, 291]]
Longitud del n-grama más largo: 84
Total de secuencias generadas: 34066


### Agrega PAD para tener secuencias de la misma longitud

In [32]:
input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')
)

# Predictores: todas las columnas menos la última. Etiqueta: la última.
predictors, label = input_sequences[:, :-1], input_sequences[:, -1]

print('input_sequences:', input_sequences.shape)
print('predictors      :', predictors.shape)
print('label           :', label.shape)

input_sequences: (34066, 84)
predictors      : (34066, 83)
label           : (34066,)


### Convierte etiquetas a 1-hot

In [33]:
label = to_categorical(label, num_classes=total_words)
print('label one-hot:', label.shape)

label one-hot: (34066, 7657)


## <span style="color:blue">Crea el modelo</span>

Cambios respecto al original:
* Se elimina `input_length` (obsoleto en Keras 3).
* Se usa `total_words // 2` (división entera) en la capa Dense intermedia.

In [34]:
model = Sequential()
model.add(Embedding(total_words, 128))   # sin input_length (deprecated)
model.add(Bidirectional(LSTM(256, return_sequences=True)))
model.add(Dropout(0.2))
model.add(LSTM(256))
model.add(Dense(total_words // 2, activation='relu',
                kernel_regularizer=regularizers.l2(0.01)))  # // evita float
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.build()
print(model.summary())



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [35]:
try:
    plot_model(model, to_file='../Imagenes/novela_model.png', show_shapes=True)
    print('Diagrama guardado en ../Imagenes/novela_model.png')
except Exception as e:
    print('No se pudo generar el diagrama:', e)

You must install pydot (`pip install pydot`) for `plot_model` to work.
Diagrama guardado en ../Imagenes/novela_model.png


## <span style="color:blue">Entrena</span>

Se usan 30 épocas por defecto. Ajusta según tiempo y resultados observados.

In [ ]:
history = model.fit(predictors, label, epochs=200, verbose=1)

Epoch 1/200
1065/1065 ━━━━━━━━━━━━━━━━━━━━ 34s 27ms/step - accuracy: 0.0429 - loss: 7.5189
Epoch 2/200
1065/1065 ━━━━━━━━━━━━━━━━━━━━ 30s 28ms/step - accuracy: 0.0503 - loss: 6.5170
Epoch 3/200
1065/1065 ━━━━━━━━━━━━━━━━━━━━ 30s 28ms/step - accuracy: 0.0672 - loss: 6.3266
Epoch 4/200
1065/1065 ━━━━━━━━━━━━━━━━━━━━ 30s 28ms/step - accuracy: 0.0719 - loss: 6.2173
Epoch 5/200
1065/1065 ━━━━━━━━━━━━━━━━━━━━ 30s 29ms/step - accuracy: 0.0746 - loss: 6.1235
Epoch 6/200
1065/1065 ━━━━━━━━━━━━━━━━━━━━ 30s 29ms/step - accuracy: 0.0806 - loss: 6.0315
Epoch 7/200
1065/1065 ━━━━━━━━━━━━━━━━━━━━ 30s 28ms/step - accuracy: 0.0795 - loss: 5.9272
Epoch 8/200
1065/1065 ━━━━━━━━━━━━━━━━━━━━ 30s 28ms/step - accuracy: 0.0854 - loss: 5.8080
Epoch 9/200
1065/1065 ━━━━━━━━━━━━━━━━━━━━ 30s 29ms/step - accuracy: 0.0955 - loss: 5.6741
Epoch 10/200
1065/1065 ━━━━━━━━━━━━━━━━━━━━ 30s 28ms/step - accuracy: 0.1079 - loss: 5.5786
Epoch 11/200
1065/1065 ━━━━━━━━━━━━━━━━━━━━ 30s 28ms/step - accuracy: 0.1264 - loss: 5.47

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



1065/1065 ━━━━━━━━━━━━━━━━━━━━ 54s 50ms/step - accuracy: 0.8523 - loss: 0.8846
Epoch 173/200
1065/1065 ━━━━━━━━━━━━━━━━━━━━ 55s 52ms/step - accuracy: 0.8522 - loss: 0.8889
Epoch 174/200
 855/1065 ━━━━━━━━━━━━━━━━━━━━ 10s 52ms/step - accuracy: 0.8528 - loss: 0.8706

In [ ]:
import matplotlib.pyplot as plt

acc  = history.history['accuracy']
loss = history.history['loss']
epochs_range = range(len(acc))

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, 'b', label='Exactitud')
plt.title('Exactitud de entrenamiento')
plt.xlabel('Época'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, 'r', label='Pérdida')
plt.title('Pérdida en entrenamiento')
plt.xlabel('Época'); plt.legend()

plt.tight_layout()
plt.show()

### (Opcional) Guarda el modelo
En Keras 3 usar extensión `.keras` o `.h5`.

In [ ]:
os.makedirs('../Saved_Models', exist_ok=True)
model.save('../Saved_Models/rnn_novela_200.keras')
print('Modelo guardado en ../Saved_Models/rnn_novela_200.keras')

In [ ]:
from tensorflow.keras.models import load_model
model = load_model('../Saved_Models/rnn_novela_200.keras')

## <span style="color:blue">Genera texto automáticamente</span>

In [ ]:
def generar_texto(seed_text, next_words=100):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list],
                                   maxlen=max_sequence_len - 1,
                                   padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]

        output_word = ''
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break

        seed_text += ' ' + output_word
    return seed_text

In [ ]:
seed_text = "Lituma preguntó a Tomas sobre"
texto_generado = generar_texto(seed_text, next_words=100)
print(texto_generado)